In [1]:
pip install pandas faiss-cpu sentence-transformers openai

   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB 330.3 kB/s eta 0:00:46
   ---------------------------------------- 0.0/15.0 MB 330.3 kB/s eta 0:00:46
   ---------------------------------------- 0.0/15.0 MB 131.3 kB/s eta 0:01:55
   ---------------------------------------- 0.0/15.0 MB 164.3 kB/s eta 0:01:32
   ---------------------------------------- 0.0/15.0 MB 164.3 kB/s eta 0:01:32
   ---------------------------------------- 0.1/15.0 MB 182.2 kB/s eta 0:01:23
   ---------------------------------------- 0.1/15.0 MB 218.5 kB/s eta 0:01:09
   ---------------------------------------- 0.1/15.0 MB 228.2 kB/s eta 0:01:06
   ---------------------------------------- 0.1/15.0 MB 257.8 kB/s eta 0:00:58
   ---------------------------------------- 0.1/15.0 MB 257.8 kB/s eta 0:00:58
   ---------------------------------------- 0.2/15.0 MB 278.4 kB/s eta

In [9]:
pip install --upgrade openai

   ---------------------------------------- 0.0/725.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/725.5 kB ? eta -:--:--
    --------------------------------------- 10.2/725.5 kB ? eta -:--:--
   - ------------------------------------- 30.7/725.5 kB 325.1 kB/s eta 0:00:03
   -- ------------------------------------ 41.0/725.5 kB 245.8 kB/s eta 0:00:03
   --- ----------------------------------- 61.4/725.5 kB 297.7 kB/s eta 0:00:03
   ---- ---------------------------------- 81.9/725.5 kB 327.3 kB/s eta 0:00:02
   ----- -------------------------------- 112.6/725.5 kB 363.1 kB/s eta 0:00:02
   ------ ------------------------------- 122.9/725.5 kB 379.3 kB/s eta 0:00:02
   -------- ----------------------------- 163.8/725.5 kB 446.5 kB/s eta 0:00:02
   ---------- --------------------------- 194.6/725.5 kB 471.4 kB/s eta 0:00:02
   ------------- ------------------------ 256.0/725.5 kB 542.5 kB/s eta 0:00:01
   --------------- ---------------------- 286.7/725.5 kB 553.0 kB/

In [83]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from openai import OpenAI
from dotenv import load_dotenv

# === Step 0: Load environment variable ===
client = OpenAI(api_key="Your_OpenAI_Key")


# === Step 1: Load Dataset ===
EXCEL_PATH = "devopsmsa.xlsx"
df = pd.read_excel(EXCEL_PATH)
df.columns = df.columns.str.strip()
df.dropna(subset=["devops_concept", "microservice_architecture_concept"], inplace=True)
df.reset_index(drop=True, inplace=True)

# === Step 2: Create Embeddings ===
model = SentenceTransformer("all-MiniLM-L6-v2")
devops_list = df["devops_concept"].tolist()
msa_list = df["microservice_architecture_concept"].tolist()

devops_embeddings = model.encode(devops_list, convert_to_tensor=True)
msa_embeddings = model.encode(msa_list, convert_to_tensor=True)

# === Step 3: Classification Logic with Threshold ===
def classify_input(user_input, threshold=0.4):
    query_embedding = model.encode(user_input, convert_to_tensor=True)
    devops_hit = util.semantic_search(query_embedding, devops_embeddings, top_k=1)[0][0]
    msa_hit = util.semantic_search(query_embedding, msa_embeddings, top_k=1)[0][0]

    devops_sim = devops_hit['score']
    msa_sim = msa_hit['score']

    if max(devops_sim, msa_sim) < threshold:
        return "unknown", devops_sim, msa_sim

    return ("devops" if devops_sim >= msa_sim else "msa"), devops_sim, msa_sim

# === Step 4: Recommendation Logic ===
def recommend_cross_domain_concepts(user_input, top_k=5):
    input_type, devops_sim, msa_sim = classify_input(user_input)

    if input_type == "unknown":
        return input_type, f"❗ The input concept '{user_input}' is not recognized as related to DevOps or MSA.\n(Similarity scores – DevOps: {devops_sim:.2f}, MSA: {msa_sim:.2f})"

    query_embedding = model.encode(user_input, convert_to_tensor=True)

    if input_type == "devops":
        hits = util.semantic_search(query_embedding, devops_embeddings, top_k=top_k)[0]
        examples = [f"{df.iloc[hit['corpus_id']]['devops_concept']} -> {df.iloc[hit['corpus_id']]['microservice_architecture_concept']}" for hit in hits]
        prompt_header = "map DevOps concepts to Microservice Architecture (MSA) concepts"
        direction = "MSA"
    else:
        hits = util.semantic_search(query_embedding, msa_embeddings, top_k=top_k)[0]
        examples = [f"{df.iloc[hit['corpus_id']]['microservice_architecture_concept']} -> {df.iloc[hit['corpus_id']]['devops_concept']}" for hit in hits]
        prompt_header = "map Microservice Architecture (MSA) concepts to DevOps concepts"
        direction = "DevOps"

    examples_text = "\n".join(examples)
    prompt = f"""
You are an assistant helping to {prompt_header}.

Here are some known mappings:
{examples_text}

Now, given this concept:
{user_input}

Suggest the top {top_k} most relevant {direction} concepts based on the examples above.

Answer:
""".strip()

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    return input_type, response.choices[0].message.content.strip()

# === Step 5: Interactive Use ===
if __name__ == "__main__":
    user_concept = input("🔍 Enter a DevOps or MSA concept: ")
    concept_type, answer = recommend_cross_domain_concepts(user_concept)
    if concept_type == "unknown":
        print(f"\n❌ Unknown Concept:\n{answer}")
    else:
        print(f"\n🧠 You entered a {concept_type.upper()} concept. Recommended related concepts:\n{answer}")


🔍 Enter a DevOps or MSA concept:  service discovery



🧠 You entered a MSA concept. Recommended related concepts:
1. Declarative Configuration Management
2. Container-Based Deployment
3. Automated DevOps Decision Making
4. Continuous Integration
5. Continuous Deployment
